In [ ]:
!pip install transformers torch pandas numpy scikit-learn tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score
import datetime

# ==========================================
# 0. 全域設定與超參數
# ==========================================
MODEL_NAME = "shibing624/macbert4zh-base" # 建議先用 base 跑通，決賽再換 large
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
OUTPUT_DIR = "./folds_data/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 假設你已經跑完前面的 Phase B，並有以下兩個檔案
# 若沒有，請先手動建立兩個簡單的 CSV 來測試此管線
TRAIN_FILE = f"{OUTPUT_DIR}train_fold_1.csv"
VAL_FILE = f"{OUTPUT_DIR}val_fold_1.csv"

# ==========================================
# 1. 資料前處理與 Dataset (包含 Head+Tail 截斷)
# ==========================================
class ESG_MTL_Dataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        
        self.head_len = int((max_len - 2) * 0.25)
        self.tail_len = (max_len - 2) - self.head_len

        # 標籤映射
        self.t1_map = {"No": 0, "Yes": 1}
        self.t2_map = {"already": 0, "within_2_years": 1, "between_2_and_5_years": 2, "longer_than_5_years": 3, "more_than_5_years": 3}
        self.t3_map = {"No": 0, "Yes": 1}
        self.t4_map = {"Clear": 0, "Not Clear": 1, "Misleading": 2}

    def _process_esg_type(self, esg_val):
        if pd.isna(esg_val) or str(esg_val).strip() == "": return "[ESG_UNK]"
        return " ".join([f"[ESG_{t.strip()}]" for t in str(esg_val).split(';')])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. 文本拼接
        esg_prefix = self._process_esg_type(row.get('esg_type', ''))
        raw_text = str(row['data'])
        full_text = f"{esg_prefix} 文本內容：{raw_text}"

        # 2. Token-level 雙向截斷
        tokens = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(tokens) > (self.max_len - 2):
            tokens = tokens[:self.head_len] + tokens[-self.tail_len:]
            
        input_ids = [self.tokenizer.cls_token_id] + tokens + [self.tokenizer.sep_token_id]
        attention_mask = [1] * len(input_ids)

        # 3. Padding
        pad_len = self.max_len - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len

        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long)
        }

        # 推論模式不回傳 Labels
        if self.is_test: return item

        # 4. 標籤轉換 (若為 N/A 或缺失，設為 -1 供 Loss 函數忽略)
        item['t1_label'] = torch.tensor(self.t1_map.get(str(row.get('promise_status')), -1), dtype=torch.float)
        item['t2_label'] = torch.tensor(self.t2_map.get(str(row.get('verification_timeline')), -1), dtype=torch.long)
        item['t3_label'] = torch.tensor(self.t3_map.get(str(row.get('evidence_status')), -1), dtype=torch.float)
        item['t4_label'] = torch.tensor(self.t4_map.get(str(row.get('evidence_quality')), -1), dtype=torch.long)

        return item

# ==========================================
# 2. 統一多任務模型架構 (MTL Backbone + 4 Heads)
# ==========================================
class ESG_Unified_MTL_Model(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # T1 & T3 (二元分類): Multi-Sample Dropout 防禦過擬合
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.t1_head = nn.Linear(hidden_size, 1)
        self.t3_head = nn.Linear(hidden_size, 1)
        
        # T2 (4分類) & T4 (3分類)
        self.t2_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, 4))
        self.t4_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, 3))

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :] # 提取 [CLS] 向量
        
        # Multi-Sample Dropout 平均化
        t1_logits = torch.mean(torch.stack([self.t1_head(d(cls_output)) for d in self.dropouts]), dim=0).squeeze(-1)
        t3_logits = torch.mean(torch.stack([self.t3_head(d(cls_output)) for d in self.dropouts]), dim=0).squeeze(-1)
        
        t2_logits = self.t2_head(cls_output)
        t4_logits = self.t4_head(cls_output)
        
        return t1_logits, t2_logits, t3_logits, t4_logits

# ==========================================
# 3. 損失函數與屏蔽機制 (Masked Loss)
# ==========================================
def calculate_mtl_loss(preds, labels):
    t1_pred, t2_pred, t3_pred, t4_pred = preds
    t1_lbl, t2_lbl, t3_lbl, t4_lbl = labels

    # T1 損失 (二元)
    bce_loss = nn.BCEWithLogitsLoss()
    loss_t1 = bce_loss(t1_pred, t1_lbl)

    # 動態遮罩：找出 T1 真實為 Yes 的樣本，T2~T4 只對這些樣本計算 Loss
    valid_mask = (t1_lbl == 1)
    
    loss_t2 = loss_t3 = loss_t4 = torch.tensor(0.0, device=DEVICE)
    
    if valid_mask.sum() > 0:
        # T2 損失 (多分類，忽略 -1)
        ce_loss_t2 = nn.CrossEntropyLoss(ignore_index=-1)
        loss_t2 = ce_loss_t2(t2_pred[valid_mask], t2_lbl[valid_mask])
        
        # T3 損失 (二元，手動過濾 -1)
        t3_valid = valid_mask & (t3_lbl != -1)
        if t3_valid.sum() > 0:
            loss_t3 = bce_loss(t3_pred[t3_valid], t3_lbl[t3_valid])
            
        # T4 損失 (多分類，針對 Misleading 加權)
        t4_valid = valid_mask & (t4_lbl != -1)
        if t4_valid.sum() > 0:
            # 第一性原理：對付極端不平衡，給 Clear(0) 較低權重，給 Misleading(2) 極高權重
            t4_weights = torch.tensor([1.0, 2.0, 5.0], device=DEVICE) 
            ce_loss_t4 = nn.CrossEntropyLoss(weight=t4_weights, ignore_index=-1)
            loss_t4 = ce_loss_t4(t4_pred[t4_valid], t4_lbl[t4_valid])

    # 任務權重調配 (基於競賽配分)
    total_loss = 0.2 * loss_t1 + 0.15 * loss_t2 + 0.3 * loss_t3 + 0.35 * loss_t4
    return total_loss

# ==========================================
# 4. 訓練流程與推論管線
# ==========================================
def train_and_predict():
    print("🚀 初始化 Tokenizer 與 模型...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    special_tokens = {'additional_special_tokens': ['[ESG_E]', '[ESG_S]', '[ESG_G]', '[ESG_UNK]']}
    tokenizer.add_special_tokens(special_tokens)
    
    model = ESG_Unified_MTL_Model(MODEL_NAME)
    model.backbone.resize_token_embeddings(len(tokenizer))
    model.to(DEVICE)
    
    # 讀取資料 (這裡假設 CSV 已經存在，若無請先跳過訓練段落)
    if os.path.exists(TRAIN_FILE) and os.path.exists(VAL_FILE):
        train_df = pd.read_csv(TRAIN_FILE)
        train_dataset = ESG_MTL_Dataset(train_df, tokenizer, MAX_LEN)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        
        print("🔥 開始訓練...")
        model.train()
        for epoch in range(1): # 示範只跑 1 Epoch
            total_loss = 0
            for batch in train_loader:
                optimizer.zero_grad()
                
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = (batch['t1_label'].to(DEVICE), batch['t2_label'].to(DEVICE), 
                          batch['t3_label'].to(DEVICE), batch['t4_label'].to(DEVICE))
                
                preds = model(input_ids, attention_mask)
                loss = calculate_mtl_loss(preds, labels)
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")
            
        # 儲存本地權重
        torch.save(model.state_dict(), "best_mtl_model.pth")
        print("✅ 模型權重已儲存。")
    else:
        print("⚠️ 找不到訓練資料，跳過訓練，直接進入推論邏輯展示。")
        
    # --- 最終目標：推論未來新資料並輸出 CSV ---
    print("\n🔮 準備執行新資料推論...")
    
    # 模擬一筆官方發布的未來新資料 (測試集)
    test_data = {
        'id': [99991, 99992],
        'esg_type': ['E', 'S;G'],
        'data': ['本公司承諾於 2030 年達成 100% 綠電使用，目前已導入太陽能版，預計每年減碳 10%。', 
                 '我們致力於促進員工福祉，但具體計畫還在研議中。']
    }
    test_df = pd.DataFrame(test_data)
    test_dataset = ESG_MTL_Dataset(test_df, tokenizer, MAX_LEN, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model.eval()
    results = []
    
    # 反向映射字典
    inv_t1 = {0: "No", 1: "Yes"}
    inv_t2 = {0: "already", 1: "within_2_years", 2: "between_2_and_5_years", 3: "longer_than_5_years"}
    inv_t3 = {0: "No", 1: "Yes"}
    inv_t4 = {0: "Clear", 1: "Not Clear", 2: "Misleading"}

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            
            t1_log, t2_log, t3_log, t4_log = model(input_ids, attention_mask)
            
            # 轉換為類別預測
            t1_preds = (torch.sigmoid(t1_log) > 0.5).long().cpu().numpy()
            t2_preds = torch.argmax(t2_log, dim=1).cpu().numpy()
            t3_preds = (torch.sigmoid(t3_log) > 0.5).long().cpu().numpy()
            t4_preds = torch.argmax(t4_log, dim=1).cpu().numpy()
            
            # 路由邏輯 (Inference Routing)
            for i in range(len(t1_preds)):
                if t1_preds[i] == 0:
                    results.append({"promise_status": "No", "verification_timeline": "N/A", "evidence_status": "N/A", "evidence_quality": "N/A"})
                else:
                    t2_res = inv_t2[t2_preds[i]]
                    if t3_preds[i] == 0:
                        results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "No", "evidence_quality": "N/A"})
                    else:
                        results.append({"promise_status": "Yes", "verification_timeline": t2_res, "evidence_status": "Yes", "evidence_quality": inv_t4[t4_preds[i]]})

    # 合併 ID 並輸出
    output_df = pd.DataFrame({'id': test_df['id']})
    pred_df = pd.DataFrame(results)
    final_output = pd.concat([output_df, pred_df], axis=1)
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    output_filename = f"output_{timestamp}.csv"
    final_output.to_csv(output_filename, index=False)
    print(f"🎉 推論完成！結果已儲存至 {output_filename}")
    print(final_output)

if __name__ == "__main__":
    train_and_predict()